In [ ]:
import sys, os, time, warnings, collections
warnings.filterwarnings("ignore")
REPO = os.path.expanduser("~/Desktop/recsys-auto-features")
sys.path.insert(0, REPO)                      # чтобы работал import src

import numpy as np, pandas as pd
from src.data import *
from src.features import window
from src.metrics import report, rank_metrics, paired_bootstrap
from src.models import fit_eval

t0 = time.time()
listens = load_yambda("listens.parquet")
likes   = load_yambda("likes.parquet")
L, K = sample_users(listens, likes, n_users=5000, seed=42)     # те же юзеры, что и раньше (seed=42)
del listens, likes
L = make_labels(L, K)
maps = {"artist_id": load_mapping("artist_item_mapping.parquet"),
        "album_id":  load_mapping("album_item_mapping.parquet")}
L, K = add_groups(L, K, maps)
t_val, t_test, t_end = time_split(L)
print("сырые данные готовы:", L.shape, K.shape, round(time.time() - t0), "сек")

Xtr = pd.read_parquet(f"{REPO}/data/Xtr_5k.parquet")
Xva = pd.read_parquet(f"{REPO}/data/Xva_5k.parquet")
Xte = pd.read_parquet(f"{REPO}/data/Xte_5k.parquet")
print("таблицы:", Xtr.shape, Xva.shape, Xte.shape)

ID, IND, DEAD = ("uid", "item_id", "label", "artist_id", "album_id"), ("uar_", "ual_"), ("ui_likes_",)
cols = [c for c in Xtr.columns if c not in ID and not c.startswith(DEAD)]
base         = ["is_organic"] + [c for c in cols if c.endswith("_all") and not c.startswith(IND)]
feats_manual = [c for c in cols if not c.startswith(IND)]
feats_ind    = cols
print(len(base), len(feats_manual), len(feats_ind), "признаков")      # должно быть 10 37 61

m_min, _ = fit_eval("minimal (all-time only)", base, Xtr, Xva, Xte)
m_man, _ = fit_eval("full manual", feats_manual, Xtr, Xva, Xte)
m_ind, _ = fit_eval("full manual + artist/album", feats_ind, Xtr, Xva, Xte)

In [ ]:
from datasets import load_dataset

ds = load_dataset("yandex/yambda", data_dir="flat/50m", data_files="likes.parquet")


In [ ]:
df = ds["train"].to_pandas()
print(df.shape)
print(df.head())
print(df.dtypes)
print(df.columns)

In [ ]:
print(df.uid.nunique(), df.item_id.nunique())
print(df.timestamp.min(), df.timestamp.max())
print(df.is_organic.mean())
print(df.groupby("uid").size().describe())
print(df.duplicated(["uid", "item_id"]).mean())  # повторные лайки одного трека
listens = load_dataset("yandex/yambda", data_dir="flat/50m", data_files="listens.parquet")["train"].to_pandas()
print(listens.shape); print(listens.head()); print(listens.dtypes)

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
users = rng.choice(listens.uid.unique(), 2000, replace=False)

L = listens[listens.uid.isin(users)].copy()
K = df[df.uid.isin(users)]            # лайки
print(L.shape, K.shape)
del listens                           # освободить память


In [ ]:
like_ts = (K.groupby(["uid", "item_id"]).timestamp.min()
             .rename("like_ts").reset_index())
L = L.merge(like_ts, on=["uid", "item_id"], how="left")

dt = L.like_ts.astype("float") - L.timestamp.astype("float")
L["label"] = ((dt >= 0) & (dt <= 3600)).astype("int8")   # окно 1 час, потом подкрутим
L = L.drop(columns="like_ts")
print(L.label.mean())

In [ ]:
T = L.timestamp
t_test = T.max() - 14*86400
t_val  = t_test - 14*86400

train = L[T < t_val]
val   = L[(T >= t_val) & (T < t_test)]
test  = L[T >= t_test]
print(len(train), len(val), len(test))

In [ ]:
import time
import pandas as pd
import numpy as np

DAY = 86400
WINDOWS = (1, 7, 30, None)

def build(cand, cutoff, windows=WINDOWS):
    HL = L[L.timestamp < cutoff]
    HK = K[K.timestamp < cutoff]

    # проверка на утечку: история строго до среза, кандидаты строго после
    assert HL.timestamp.max() < cutoff and HK.timestamp.max() < cutoff
    assert cand.timestamp.min() >= cutoff

    X = cand[["uid", "item_id", "is_organic", "label"]].copy()

    for w in windows:
        lo = 0 if w is None else cutoff - w * DAY
        l = HL[HL.timestamp >= lo]
        k = HK[HK.timestamp >= lo]
        wn = "all" if w is None else f"{w}d"

        for ent, keys in [("u", ["uid"]), ("i", ["item_id"]), ("ui", ["uid", "item_id"])]:
            for ev, h in [("listens", l), ("likes", k)]:
                col = f"{ent}_{ev}_{wn}"
                c = h.groupby(keys).size().rename(col).reset_index()
                X = X.merge(c, on=keys, how="left")
                X[col] = X[col].fillna(0)

        for ent in ["u", "i"]:
            X[f"{ent}_cr_{wn}"] = X[f"{ent}_likes_{wn}"] / (X[f"{ent}_listens_{wn}"] + 1)

        # средняя доля прослушивания в прошлом
        for ent, keys in [("u", "uid"), ("i", "item_id")]:
            col = f"{ent}_played_{wn}"
            c = l.groupby(keys).played_ratio_pct.mean().rename(col).reset_index()
            X = X.merge(c, on=keys, how="left")   # NaN оставляем: бустинг умеет с ними работать
    return X

def window(a, b):
    return L[(L.timestamp >= a) & (L.timestamp < b)]

t0 = time.time()
c_tr = t_val - 14 * DAY
Xtr = build(window(c_tr, t_val),         c_tr);   print("train", Xtr.shape, time.time() - t0)
Xva = build(window(t_val, t_test),       t_val);  print("val",   Xva.shape, time.time() - t0)
Xte = build(window(t_test, T.max() + 1), t_test); print("test",  Xte.shape, time.time() - t0)
print(Xtr.label.mean(), Xva.label.mean(), Xte.label.mean())

In [ ]:
from sklearn.metrics import roc_auc_score, log_loss

def rank_metrics(X, score, k=10):
    d = pd.DataFrame({"uid": X.uid.values, "y": X.label.values, "s": np.asarray(score, dtype="float64")})
    d["s"] += np.random.default_rng(0).random(len(d)) * 1e-9      # случайный разбор ничьих
    d = d.sort_values(["uid", "s"], ascending=[True, False])
    d["r"] = d.groupby("uid").cumcount() + 1

    npos = d.groupby("uid").y.sum()
    npos = npos[npos > 0]                                          # только юзеры с позитивом
    top = d[(d.r <= k) & d.uid.isin(npos.index)]

    dcg = (top.y / np.log2(top.r + 1)).groupby(top.uid).sum().reindex(npos.index).fillna(0)
    ideal = np.cumsum(1 / np.log2(np.arange(2, k + 2)))
    idcg = pd.Series(ideal[np.minimum(npos.values, k).astype(int) - 1], index=npos.index)
    hits = top.y.groupby(top.uid).sum().reindex(npos.index).fillna(0)
    return {f"ndcg@{k}": (dcg / idcg).mean(), f"recall@{k}": (hits / npos).mean()}

def report(name, X, score):
    r = {"AUC": roc_auc_score(X.label, score)}
    r.update(rank_metrics(X, score))
    print(f"{name:28s}", {k: round(v, 4) for k, v in r.items()})
    return r

rng = np.random.default_rng(1)
res = {}
for nm, X in [("val", Xva), ("test", Xte)]:
    res[f"random/{nm}"]  = report(f"random [{nm}]",      X, rng.random(len(X)))
    res[f"popular/{nm}"] = report(f"top-popular [{nm}]", X, X.i_likes_all.values)

    # pip install lightgbm
import lightgbm as lgb

def fit_eval(name, feats, params=None):
    p = dict(n_estimators=1000, learning_rate=0.05, num_leaves=63,
             subsample=0.8, subsample_freq=1, colsample_bytree=0.8, verbose=-1)
    p.update(params or {})
    m = lgb.LGBMClassifier(**p)
    m.fit(Xtr[feats], Xtr.label, eval_set=[(Xva[feats], Xva.label)],
          callbacks=[lgb.early_stopping(50, verbose=False)])
    out = {}
    for nm, X in [("val", Xva), ("test", Xte)]:
        out[nm] = report(f"{name} [{nm}]", X, m.predict_proba(X[feats])[:, 1])
        out[nm]["logloss"] = log_loss(X.label, m.predict_proba(X[feats])[:, 1])
    return m, out

base = ["is_organic"] + [c for c in Xtr.columns if c.endswith("_all") and c != "uid"]
m_min, r_min = fit_eval("boosting minimal", base)

In [ ]:
Xtr, Xva, Xte = [X[X.ui_likes_all == 0].reset_index(drop=True) for X in (Xtr, Xva, Xte)]
print(len(Xtr), len(Xva), len(Xte), Xtr.label.mean(), Xva.label.mean(), Xte.label.mean())

rng = np.random.default_rng(1)
for nm, X in [("val", Xva), ("test", Xte)]:
    report(f"random [{nm}]", X, rng.random(len(X)))
    report(f"top-popular [{nm}]", X, X.i_likes_all.values)

base = ["is_organic"] + [c for c in Xtr.columns if c.endswith("_all")]
m_min, r_min = fit_eval("boosting minimal", base)

all_feats = [c for c in Xtr.columns if c not in ("uid", "item_id", "label")]
m_full, r_full = fit_eval("boosting full (manual)", all_feats)

imp = pd.Series(m_full.booster_.feature_importance("gain"), index=all_feats).sort_values(ascending=False)
print(imp.head(15))

In [ ]:
def build_train(n_cuts=4, step=7):
    parts = []
    for k in range(1, n_cuts + 1):
        c = t_val - k * step * DAY
        cand = window(c, c + step * DAY)
        X = build(cand, c)
        parts.append(X[X.ui_likes_all == 0])
    return pd.concat(parts, ignore_index=True)

Xtr = build_train()
print(Xtr.shape, Xtr.label.mean())

m_full4, r_full4 = fit_eval("boosting full, 4 cuts", all_feats)

In [ ]:
from huggingface_hub import list_repo_files
print([f for f in list_repo_files("yandex/yambda", repo_type="dataset") if "mapping" in f])


In [ ]:
from datasets import load_dataset

def load_map(fname):
    return load_dataset("yandex/yambda", data_files=fname)["train"].to_pandas()

am = load_map("artist_item_mapping.parquet")
bm = load_map("album_item_mapping.parquet")
print(am.columns.tolist(), am.shape)
print(bm.columns.tolist(), bm.shape)

maps = {}
for gcol, mp in [("artist_id", am), ("album_id", bm)]:
    src = [c for c in mp.columns if c != "item_id"][0]
    maps[gcol] = mp.drop_duplicates("item_id").set_index("item_id")[src]
    L[gcol] = L.item_id.map(maps[gcol]).fillna(-1).astype("int64")
    K[gcol] = K.item_id.map(maps[gcol]).fillna(-1).astype("int64")
    print(gcol, "покрытие маппингом:", (L[gcol] >= 0).mean())

def add_indirect(X, cutoff, windows=WINDOWS):
    X = X.drop(columns=[c for c in X.columns if c.startswith(("uar_", "ual_")) or c in ("artist_id", "album_id")],
               errors="ignore")
    HL = L[L.timestamp < cutoff]
    HK = K[K.timestamp < cutoff]
    for gname, gcol in [("ar", "artist_id"), ("al", "album_id")]:
        X[gcol] = X.item_id.map(maps[gcol]).fillna(-1).astype("int64")
        for w in windows:
            lo = 0 if w is None else cutoff - w * DAY
            wn = "all" if w is None else f"{w}d"
            for ev, h in [("listens", HL[HL.timestamp >= lo]), ("likes", HK[HK.timestamp >= lo])]:
                col = f"u{gname}_{ev}_{wn}"
                c = h.groupby(["uid", gcol]).size().rename(col).reset_index()
                X = X.merge(c, on=["uid", gcol], how="left")
                X[col] = X[col].fillna(0)
            X[f"u{gname}_cr_{wn}"] = X[f"u{gname}_likes_{wn}"] / (X[f"u{gname}_listens_{wn}"] + 1)
    return X

def build_train2(n_cuts=4, step=7):
    parts = []
    for k in range(1, n_cuts + 1):
        c = t_val - k * step * DAY
        X = build(window(c, c + step * DAY), c)
        X = X[X.ui_likes_all == 0]
        parts.append(add_indirect(X, c))
    return pd.concat(parts, ignore_index=True)

t0 = time.time()
Xtr = build_train2();          print("train", Xtr.shape, time.time() - t0)
Xva = add_indirect(Xva, t_val);  Xte = add_indirect(Xte, t_test)
print(Xva.shape, Xte.shape, time.time() - t0)

In [ ]:
import warnings; warnings.filterwarnings("ignore")

# 0) сохранить таблицы, чтобы не пересчитывать после перезапуска ядра
import os
os.makedirs("data", exist_ok=True)
os.makedirs("reports", exist_ok=True)
print("рабочая папка:", os.getcwd())

Xtr.to_parquet("data/Xtr.parquet"); Xva.to_parquet("data/Xva.parquet"); Xte.to_parquet("data/Xte.parquet")
ID = ("uid", "item_id", "label", "artist_id", "album_id")
IND = ("uar_", "ual_")
base         = ["is_organic"] + [c for c in Xtr.columns if c.endswith("_all") and c not in ID and not c.startswith(IND)]
feats_manual = [c for c in Xtr.columns if c not in ID and not c.startswith(IND)]
feats_ind    = [c for c in Xtr.columns if c not in ID]
print(len(base), len(feats_manual), len(feats_ind), "признаков")
print("доли позитивов:", Xtr.label.mean(), Xva.label.mean(), Xte.label.mean())

rows = []
def log(name, out):
    for nm in ("val", "test"):
        rows.append({"model": name, "split": nm, **out[nm]})

# 1) бейзлайны на отфильтрованных данных
rng = np.random.default_rng(1)
for nm, X in [("val", Xva), ("test", Xte)]:
    r = report(f"random [{nm}]", X, rng.random(len(X)));            rows.append({"model": "random", "split": nm, **r})
    r = report(f"top-popular [{nm}]", X, X.i_likes_all.values);     rows.append({"model": "top-popular", "split": nm, **r})

# 2) три бустинга
m_min, r = fit_eval("minimal (all-time only)", base);              log("minimal (all-time only)", r)
m_man, r = fit_eval("full manual", feats_manual);                  log("full manual", r)
m_ind, r = fit_eval("full manual + artist/album", feats_ind);      log("full manual + artist/album", r)

res = pd.DataFrame(rows).round(4)
print(res.to_string(index=False))
res.to_csv("reports/results_manual.csv", index=False)

imp = pd.Series(m_ind.booster_.feature_importance("gain"), index=feats_ind).sort_values(ascending=False)
print(imp.head(15))

In [ ]:
def paired_bootstrap(X, scores, ref, n=100, k=10, seed=0):
    rng = np.random.default_rng(seed)
    d = X[["uid", "label"]].reset_index(drop=True).copy()
    for nm, s in scores.items():
        d[nm] = np.asarray(s)
    groups = {u: g for u, g in d.groupby("uid")}
    users = np.array(list(groups))
    diffs = {nm: [] for nm in scores if nm != ref}
    for _ in range(n):
        pick = rng.choice(users, len(users))
        Xb = pd.concat([groups[u].assign(uid=i) for i, u in enumerate(pick)], ignore_index=True)
        base_v = rank_metrics(Xb, Xb[ref].values)[f"ndcg@{k}"]
        for nm in diffs:
            diffs[nm].append(rank_metrics(Xb, Xb[nm].values)[f"ndcg@{k}"] - base_v)
    for nm, v in diffs.items():
        lo, hi = np.percentile(v, [2.5, 97.5])
        print(f"{nm:14s} - {ref}: среднее {np.mean(v):+.4f}, 95% интервал [{lo:+.4f}, {hi:+.4f}]")

S = {"manual":       m_man.predict_proba(Xte[feats_manual])[:, 1],
     "minimal":      m_min.predict_proba(Xte[base])[:, 1],
     "manual+ar/al": m_ind.predict_proba(Xte[feats_ind])[:, 1],
     "random":       np.random.default_rng(5).random(len(Xte))}
paired_bootstrap(Xte, S, ref="manual")

In [ ]:
import sys, os, time, warnings
warnings.filterwarnings("ignore")
REPO = os.path.expanduser("~/Desktop/recsys-auto-features")
sys.path.insert(0, REPO)

import numpy as np, pandas as pd
from src.data import *
from src.features import *
from src.metrics import report, rank_metrics, paired_bootstrap
from src.models import fit_eval, feature_columns

N_USERS = 5000
t0 = time.time()

listens = load_yambda("listens.parquet")
likes   = load_yambda("likes.parquet")
n_all = listens.uid.nunique()
print("всего юзеров в датасете:", n_all)

L, K = sample_users(listens, likes, n_users=min(N_USERS, n_all), seed=42)
del listens, likes
L = make_labels(L, K)
maps = {"artist_id": load_mapping("artist_item_mapping.parquet"),
        "album_id":  load_mapping("album_item_mapping.parquet")}
L, K = add_groups(L, K, maps)
t_val, t_test, t_end = time_split(L)
print("данные готовы:", L.shape, K.shape, round(time.time() - t0), "сек")

def shrink(X):
    f = X.select_dtypes("float64").columns
    X[f] = X[f].astype("float32")
    return X

def eval_set(a, b, cutoff):
    X = build(L, K, window(L, a, b), cutoff)
    X = X[X.ui_likes_all == 0]                 # тот же фильтр, что и в train
    return shrink(add_indirect(L, K, X, cutoff, maps))

Xtr = shrink(build_train(L, K, t_val, maps=maps))
print("train:", Xtr.shape, round(time.time() - t0), "сек")
Xva = eval_set(t_val, t_test, t_val)
Xte = eval_set(t_test, t_end + 1, t_test)
print("val/test:", Xva.shape, Xte.shape, round(time.time() - t0), "сек")
print("доли позитивов:", Xtr.label.mean(), Xva.label.mean(), Xte.label.mean())

os.makedirs(f"{REPO}/data", exist_ok=True)       # data/ в .gitignore, на GitHub не уйдёт
for nm, X in [("Xtr", Xtr), ("Xva", Xva), ("Xte", Xte)]:
    X.to_parquet(f"{REPO}/data/{nm}_5k.parquet")


In [ ]:
ID  = ("uid", "item_id", "label", "artist_id", "album_id")
IND = ("uar_", "ual_")
DEAD = ("ui_likes_",)      # после фильтра эти признаки всегда 0

cols = [c for c in Xtr.columns if c not in ID and not c.startswith(DEAD)]
base         = ["is_organic"] + [c for c in cols if c.endswith("_all") and not c.startswith(IND)]
feats_manual = [c for c in cols if not c.startswith(IND)]
feats_ind    = cols
print(len(base), len(feats_manual), len(feats_ind), "признаков")

rows = []
def log(name, out):
    for nm in ("val", "test"):
        rows.append({"model": name, "split": nm, **out[nm]})

rng = np.random.default_rng(1)
for nm, X in [("val", Xva), ("test", Xte)]:
    rows.append({"model": "random", "split": nm, **report(f"random [{nm}]", X, rng.random(len(X)))})
    rows.append({"model": "top-popular", "split": nm, **report(f"top-popular [{nm}]", X, X.i_likes_all.values)})

m_min, r = fit_eval("minimal (all-time only)", base, Xtr, Xva, Xte);          log("minimal (all-time only)", r)
m_man, r = fit_eval("full manual", feats_manual, Xtr, Xva, Xte);              log("full manual", r)
m_ind, r = fit_eval("full manual + artist/album", feats_ind, Xtr, Xva, Xte);  log("full manual + artist/album", r)

res = pd.DataFrame(rows).round(4)
print(res.to_string(index=False))
os.makedirs(f"{REPO}/reports", exist_ok=True)
res.to_csv(f"{REPO}/reports/results_manual_5k.csv", index=False)

S = {"manual":       m_man.predict_proba(Xte[feats_manual])[:, 1],
     "minimal":      m_min.predict_proba(Xte[base])[:, 1],
     "manual+ar/al": m_ind.predict_proba(Xte[feats_ind])[:, 1],
     "random":       np.random.default_rng(5).random(len(Xte))}
_ = paired_bootstrap(Xte, S, ref="manual", n=100)


In [ ]:
from src.auto_features import build_train_auto, build_auto

t0 = time.time()
def ev(a, b, c):
    X = build_auto(L, K, window(L, a, b), c, maps)
    return X[X.ui_likes_all == 0]

Xtr_a = build_train_auto(L, K, t_val, maps);   print("train", Xtr_a.shape, round(time.time()-t0), "сек")
Xva_a = ev(t_val, t_test, t_val)
Xte_a = ev(t_test, t_end + 1, t_test);         print("val/test", Xva_a.shape, Xte_a.shape, round(time.time()-t0), "сек")

# строки должны совпадать с прежним тестом, иначе bootstrap бессмыслен
assert (Xte_a.uid.values == Xte.uid.values).all() and (Xte_a.item_id.values == Xte.item_id.values).all()

ID = ("uid", "item_id", "label", "artist_id", "album_id")
auto_cols = [c for c in Xtr_a.columns if c not in ID and not c.startswith(("ui_likes_", "ui_cr_"))]
print(len(auto_cols), "автоматических признаков")

m_all, _ = fit_eval("auto: все признаки", auto_cols, Xtr_a, Xva_a, Xte_a)

imp = pd.Series(m_all.booster_.feature_importance("gain"), index=auto_cols).sort_values(ascending=False)
sel = list(imp.index[:37])
m_sel, _ = fit_eval("auto: топ-37", sel, Xtr_a, Xva_a, Xte_a)

import collections
print("окна среди выбранных:", collections.Counter(c.rsplit("_", 1)[1] for c in sel).most_common())
print("сущности среди выбранных:", collections.Counter(c.split("_")[0] for c in sel).most_common())
print(imp.head(15))

S = {"manual":    m_man.predict_proba(Xte[feats_manual])[:, 1],
     "auto-all":  m_all.predict_proba(Xte_a[auto_cols])[:, 1],
     "auto-top37": m_sel.predict_proba(Xte_a[sel])[:, 1],
     "random":    np.random.default_rng(5).random(len(Xte))}
_ = paired_bootstrap(Xte, S, ref="manual", n=100)

In [ ]:
def prune_by_corr(X, ranked, k, thr=0.95, sample=200_000, seed=0):
    """Идём по списку по убыванию важности; берём признак, если его |корреляция рангов| с каждым уже взятым < thr."""
    S = X[ranked].sample(min(sample, len(X)), random_state=seed).rank()      # ранги -> корреляция Спирмена
    C = S.corr().abs().values
    pos = {c: i for i, c in enumerate(ranked)}
    chosen = []
    for c in ranked:
        if all(C[pos[c], pos[s]] < thr for s in chosen):
            chosen.append(c)
        if len(chosen) == k:
            break
    return chosen

def ent(c): return "organic" if c == "is_organic" else c.split("_")[0]

ranked = list(imp.index)
results = {}
for thr in (0.99, 0.95, 0.90):
    sel_p = prune_by_corr(Xtr_a, ranked, 37, thr)
    m, out = fit_eval(f"auto: топ-37, корр < {thr}", sel_p, Xtr_a, Xva_a, Xte_a)
    results[thr] = (sel_p, m, out)
    print("  сущности:", collections.Counter(ent(c) for c in sel_p).most_common())
    print("  окна:", collections.Counter(c.rsplit('_', 1)[1] for c in sel_p).most_common(), "\n")

best_thr = max(results, key=lambda t: results[t][2]["val"]["ndcg@10"])      # выбор по val, не по test
print("лучший порог по val:", best_thr)
sel_best, m_best, _ = results[best_thr]

S2 = {"manual":       m_man.predict_proba(Xte[feats_manual])[:, 1],
      "manual+ar/al": m_ind.predict_proba(Xte[feats_ind])[:, 1],
      "auto-all":     m_all.predict_proba(Xte_a[auto_cols])[:, 1],
      "auto-top37":   m_sel.predict_proba(Xte_a[sel])[:, 1],
      "auto-pruned":  m_best.predict_proba(Xte_a[sel_best])[:, 1]}
_ = paired_bootstrap(Xte, S2, ref="manual", n=100)

In [ ]:
import sys, os, time, warnings, collections
warnings.filterwarnings("ignore")
REPO = os.path.expanduser("~/Desktop/recsys-auto-features")
sys.path.insert(0, REPO)
import numpy as np, pandas as pd, lightgbm as lgb
from src.metrics import report, rank_metrics, paired_bootstrap
from src.models import fit_eval

D = f"{REPO}/data"
Xtr, Xva, Xte = [pd.read_parquet(f"{D}/{n}_5k.parquet") for n in ("Xtr", "Xva", "Xte")]
Xtr_a, Xva_a, Xte_a = [pd.read_parquet(f"{D}/{n}_a_5k.parquet") for n in ("Xtr", "Xva", "Xte")]
assert (Xte_a.uid.values == Xte.uid.values).all() and (Xte_a.item_id.values == Xte.item_id.values).all()

ID, IND, DEAD = ("uid", "item_id", "label", "artist_id", "album_id"), ("uar_", "ual_"), ("ui_likes_",)
cols = [c for c in Xtr.columns if c not in ID and not c.startswith(DEAD)]
feats_manual = [c for c in cols if not c.startswith(IND)]
auto_cols = [c for c in Xtr_a.columns if c not in ID and not c.startswith(("ui_likes_", "ui_cr_"))]
print(len(feats_manual), len(auto_cols), "признаков")            # должно быть 37 151

m_man, _ = fit_eval("full manual", feats_manual, Xtr, Xva, Xte)   # нужен для строки CLF в bootstrap

In [ ]:
def grp(X):
    """Размеры групп = подряд идущие строки одного юзера."""
    u = X.uid.values
    edges = np.flatnonzero(np.r_[True, u[1:] != u[:-1], True])
    return np.diff(edges)

assert len(grp(Xva_a)) == Xva_a.uid.nunique(), "val не сгруппирован по uid"
assert len(grp(Xte_a)) == Xte_a.uid.nunique(), "test не сгруппирован по uid"
assert Xtr_a.uid.nunique() <= len(grp(Xtr_a)) <= 4 * Xtr_a.uid.nunique(), "train не сгруппирован"
print("групп train/val/test:", len(grp(Xtr_a)), len(grp(Xva_a)), len(grp(Xte_a)))

def fit_rank(name, feats, Xtr, Xva, Xte, n_estimators=600):
    m = lgb.LGBMRanker(objective="lambdarank", n_estimators=n_estimators, learning_rate=0.05,
                       num_leaves=63, subsample=0.8, subsample_freq=1, colsample_bytree=0.8, verbose=-1)
    m.fit(Xtr[feats], Xtr.label, group=grp(Xtr),
          eval_set=[(Xva[feats], Xva.label)], eval_group=[grp(Xva)], eval_at=[10],
          callbacks=[lgb.early_stopping(50, verbose=False)])
    for nm, X in (("val", Xva), ("test", Xte)):
        report(f"{name} [{nm}]", X, m.predict(X[feats]))
    return m

def ent(c): return "organic" if c == "is_organic" else c.split("_")[0]

t0 = time.time()
r_man = fit_rank("RANK manual (37)", feats_manual, Xtr, Xva, Xte);          print(round(time.time() - t0), "сек")
r_all = fit_rank("RANK auto: все (151)", auto_cols, Xtr_a, Xva_a, Xte_a);   print(round(time.time() - t0), "сек")

imp_r = pd.Series(r_all.booster_.feature_importance("gain"), index=auto_cols).sort_values(ascending=False)
sel_r = list(imp_r.index[:37])
print("сущности в отборе ранкера:", collections.Counter(ent(c) for c in sel_r).most_common())
print("окна в отборе ранкера:", collections.Counter(c.rsplit("_", 1)[1] for c in sel_r).most_common())
print(imp_r.head(15))

r_sel = fit_rank("RANK auto: топ-37", sel_r, Xtr_a, Xva_a, Xte_a);          print(round(time.time() - t0), "сек")

S3 = {"RANK manual":     r_man.predict(Xte[feats_manual]),
      "RANK auto-all":   r_all.predict(Xte_a[auto_cols]),
      "RANK auto-top37": r_sel.predict(Xte_a[sel_r]),
      "CLF manual":      m_man.predict_proba(Xte[feats_manual])[:, 1]}
_ = paired_bootstrap(Xte, S3, ref="RANK manual", n=100)

In [ ]:
import json
rows = [
 # model, признаков, NDCG val, NDCG test, Recall val, Recall test (цифры из ваших выводов)
 ("random",                    0,   0.0497, 0.0575, 0.0669, 0.0806),
 ("full manual",               37,  0.0722, 0.0818, 0.0968, 0.1123),
 ("manual + artist/album",     61,  0.0777, 0.0852, 0.1064, 0.1136),
 ("auto: все признаки",        151, 0.0812, 0.0848, 0.1111, 0.1151),
 ("auto: топ-37",              37,  0.0700, 0.0785, 0.0927, 0.1077),
 ("auto: топ-37, корр < 0.99", 37,  0.0767, 0.0780, 0.1033, 0.1036),
 ("auto: топ-37, корр < 0.95", 37,  0.0742, 0.0775, 0.1028, 0.1066),
 ("auto: топ-37, корр < 0.90", 37,  0.0752, 0.0832, 0.1024, 0.1109),
]
df = pd.DataFrame(rows, columns=["model", "n_features", "ndcg10_val", "ndcg10_test", "recall10_val", "recall10_test"])
df.to_csv(f"{REPO}/reports/results_auto.csv", index=False)

bootstrap = {  # разница NDCG@10 на test против full manual, 95% интервал
  "manual + artist/album": [0.0027, -0.0047, 0.0089],
  "auto-all":              [0.0020, -0.0052, 0.0090],
  "auto-top37":            [-0.0016, -0.0088, 0.0045],
  "auto-pruned":           [-0.0044, -0.0122, 0.0029],
}
json.dump(bootstrap, open(f"{REPO}/reports/bootstrap_ndcg_test.json", "w"), ensure_ascii=False, indent=2)
print(df)